## 第一步：构建MCP服务

[↓↓mcp-python-sdk document](https://github.com/modelcontextprotocol/python-sdk)

### 1. 安装相关包
`uv add "mcp[cli]"`



### 2. python脚本

```python
import os
from mcp.server import MCPServer

mcp = MCPServer("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.resource("greeting://{name}")
def greeting(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(
        transport="streamable-http",
        host=os.environ.get("HOST", "0.0.0.0"),  
        port=int(os.environ.get("PORT", "8000")),
    )
```

### 3. 启动MCP服务

```
uv run src/mcp_playbook/01.01.server.py
      Built mcp-playbook @ file:///Users/wangquanjun/Documents/project/mcp-playbook                                                                                                                                                                                                                                                                                     Uninstalled 1 package in 2ms
Installed 1 package in 2ms
INFO:     Started server process [32941]
INFO:     Waiting for application startup.
[11/19/26 10:46:18] INFO     StreamableHTTP session manager started                                                                                                                                                                                                                                                                       streamable_http_manager.py:162
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

## Harness连接到MCP 服务

> 以Deepseek Harness为例

### 1. [启动Deepseek Harness](https://github.com/deepseek-ai/deepseek-harness/blob/master/README.md)

```
git clone https://github.com/deepseek-ai/deepseek-harness.git
cd deepseek-harness
pnpm install
pnpm run build
pnpm dsh web

$ node --import tsx/esm apps/cli/src/bin.ts web
dsh web: http://127.0.0.1:3080/?token=wE3f_fUOIM_VSsJ2grD0wp0Fhl3GjXNXWJGlFYdWd7E
dsh web: opening the default browser; pass --no-open to disable
[llm-deepseek-account] model listing from https://api.deepseek.com/models loaded 2 model(s), skipped 0 row(s)
[deepseek-account] request failed {
  path: '/auth-api/v0/users/current',
  status: 403,
  errorCode: 'network'
}
[deepseek-account] request failed {
  path: '/auth-api/v0/users/current',
  status: 403,
  errorCode: 'network'
}
[deepseek-account] request failed {
  path: '/auth-api/v0/users/current',
  status: 403,
  errorCode: 'network'
}
[deepseek-account] request failed {
  path: '/auth-api/v0/users/current',
  status: 403,
  errorCode: 'network'
}

```

浏览器打开 http://127.0.0.1:3080



#### 接口测试-initialize & 获取session_id
```
curl -sS -D /tmp/mcp_init_headers.txt -o /tmp/mcp_init_body.txt \
  -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"curl-test","version":"1.0"}}}'


cat /tmp/mcp_init_headers.txt 
HTTP/1.1 200 OK
date: Thu, 19 Nov 2026 03:05:39 GMT
server: uvicorn
cache-control: no-cache, no-transform
connection: keep-alive
content-type: text/event-stream
mcp-session-id: 56d6646cf0f548c1b08d25883411898b
x-accel-buffering: no
Transfer-Encoding: chunked


SESSION_ID=$(awk 'tolower($1)=="mcp-session-id:"{print $2}' /tmp/mcp_init_headers.txt | tr -d '\r')


echo $SESSION_ID
56d6646cf0f548c1b08d25883411898b
```

#### 接口测试-notifications/initialized

```
curl -sS -o /dev/null -w "notifications/initialized HTTP 状态: %{http_code}\n" \
  -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "mcp-session-id: ${SESSION_ID}" \
  -d '{"jsonrpc":"2.0","method":"notifications/initialized"}'
notifications/initialized HTTP 状态: 202
```


#### 接口测试-tools/list

```
curl -sS -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "mcp-session-id: ${SESSION_ID}" \
  -d '{"jsonrpc":"2.0","id":2,"method":"tools/list"}'
event: message
data: {
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "tools": [
      {
        "description": "Add two numbers.",
        "inputSchema": {
          "properties": {
            "a": {
              "title": "A",
              "type": "integer"
            },
            "b": {
              "title": "B",
              "type": "integer"
            }
          },
          "required": [
            "a",
            "b"
          ],
          "type": "object",
          "title": "addArguments"
        },
        "name": "add",     # your tools here
        "outputSchema": {
          "properties": {
            "result": {
              "title": "Result",
              "type": "integer"
            }
          },
          "required": [
            "result"
          ],
          "type": "object",
          "title": "addOutput"
        }
      }
    ]
  }
}
```
#### 接口测试-调用工具

```
 curl -sS -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "mcp-session-id: ${SESSION_ID}" \
  -d '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"add","arguments":{"a":3,"b":5}}}'
event: message
data: {"jsonrpc":"2.0","id":3,"result":{"content":[{"text":"8","type":"text"}],"isError":false,"structuredContent":{"result":8}}}
```


### 2. Harness 连接到 MCP 服务

> 最简单的方式是直接写一个配置文件，指向刚才构建的mcp服务
```
cat ~/.dsh/cordis.patch.yml
- insert:
    - id: remote-demo-mcp
      name: '@deepseek-ai/dsh-mcp-client'
      config:
        serverName: demo                # 工具名前缀 → mcp__demo__add
        transport: streamable-http
        url: http://127.0.0.1:8000/mcp  # 你的 MCP 服务地址
        # 如果后续加了鉴权，在这里添加 headers
        # headers:
        #   Authorization: !!js '`Bearer ${process.env.MCP_TOKEN}`'
```



### 3. 在Deepseek Harness WebUI调用工具



请调用 demo 服务器 的 add 工具，计算 3 加 2444

---
已完成，用时 5秒

已调用工具
上下文注入
time-context
工具调用

mcp__demo__add · {"a": 3, "b": 2444}
3 + 2444 = 2447

（通过 demo 服务器的 add 工具计算得出。）